# 0. Run PyNNLF: Model Comparison, 14 Models x 4 Datasets

## Purpose
Runs the journal article's model comparison at the 1-day forecast horizon, locally or on Databricks: the 14 models of `specs/model_comparison_fh8_batch.yaml` on ASHD (`ds20`), ACAP-EDP (`ds11`), SunDance (`ds38`) and Fluvius (`ds40`). Output goes to `experiment_result_databricks/`, the store the section-01 processing notebook reads alongside the original laptop runs in `experiment_result/`; the folder keeps its historical name whichever machine runs it.

The run is resumable and skips every combination that already has a result in either folder, so the 12 original ASHD runs and the ACAP-EDP XGBoost run are not repeated. Expected count: `4 datasets x 14 models = 56`, of which 43 are pending at the time of writing. The runner walks the model list across all four datasets, cheapest model first, so a partial run yields whole rows of the table. The LSTM is the only expensive model (about 10 hours on a 12-thread laptop for its four runs, against 8 to 10 hours for the other 39 runs together), so it has its own cell and can be run at a different time or on a different machine.

## Inputs
- `specs/model_comparison_fh8_batch.yaml`
- `data/ds20_*.csv`, `data/ds11_*.csv`, `data/ds38_*.csv`, `data/ds40_*.csv`
- `scripts/run_model_comparison_batch.py`, which holds the run logic so the same code runs from a terminal

## Run Flow
1. Discover Paths
2. Databricks Dependency Install
3. Reserve CPU Cores For Other Work
4. Load The Batch Spec And Show The Plan
5. Validate Input Datasets
6. Run Every Model Except The LSTM
7. Run The LSTM
8. Validate The Combined Recap

## Outputs
- `experiment_result_databricks/E*/` experiment folders and `experiment_result_databricks/a1_experiment_result.csv`


## 1. Discover Paths

The path logic is relative and works from Databricks Repos/Workspace or from a local checkout. It finds the repository root from `pyproject.toml` and the publication project from `publication/journal_article_1`.

In [ ]:
from pathlib import Path
import os
import sys


def databricks_notebook_candidates():
    candidates = []
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()  # noqa: F821
        nb_path = ctx.notebookPath().get()
    except Exception:
        return candidates
    raw = Path(nb_path)
    candidates.append(raw)
    if nb_path.startswith('/Workspace/'):
        candidates.append(Path(nb_path))
    else:
        candidates.append(Path('/Workspace') / nb_path.lstrip('/'))
    return candidates


def upward_candidates(start):
    start = Path(start).resolve()
    return [start, *start.parents]


def find_repo_root():
    starts = [Path.cwd(), *databricks_notebook_candidates()]
    seen = set()
    candidates = []
    for start in starts:
        try:
            for candidate in upward_candidates(start):
                key = str(candidate)
                if key not in seen:
                    seen.add(key)
                    candidates.append(candidate)
        except Exception:
            continue
    for candidate in candidates:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'pynnlf').exists():
            return candidate
    details = '\n'.join(str(c) for c in candidates[:30])
    raise FileNotFoundError('Could not locate the PyNNLF repo root. Checked:\n' + details)


REPO_ROOT = find_repo_root()
PROJECT_DIR = REPO_ROOT / 'publication' / 'journal_article_1'
if not PROJECT_DIR.exists():
    raise FileNotFoundError(f'Publication project folder not found: {PROJECT_DIR}')

os.chdir(PROJECT_DIR)
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

print(f'Repo root: {REPO_ROOT}')
print(f'Publication project: {PROJECT_DIR}')
print(f'Current working directory: {Path.cwd()}')


## 2. Databricks Dependency Install

On Databricks this installs the dependencies needed by all 14 models (PyTorch covers the LSTM and the two new models). Locally, this cell skips installation and uses your current Python environment. Set the `install_dependencies` widget to `no` on a rerun after the first install.

In [ ]:
import importlib
import subprocess


def running_on_databricks():
    if 'DATABRICKS_RUNTIME_VERSION' in os.environ:
        return True
    try:
        dbutils  # noqa: F821
        return True
    except Exception:
        return False


def get_install_dependencies(default='yes'):
    choices = ['yes', 'no']
    try:
        dbutils.widgets.dropdown('install_dependencies', default, choices, 'Install dependencies')  # noqa: F821
        value = dbutils.widgets.get('install_dependencies')  # noqa: F821
    except Exception:
        value = default
    if value not in choices:
        raise ValueError(f'Unsupported install_dependencies={value!r}; expected one of {choices}')
    return value


def pip_install(args):
    cmd = [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', *args]
    print('Running:', ' '.join(str(part) for part in cmd))
    subprocess.check_call(cmd)


INSTALL_DEPENDENCIES = get_install_dependencies()
print(f'Install dependencies: {INSTALL_DEPENDENCIES}')

if running_on_databricks() and INSTALL_DEPENDENCIES == 'yes':
    print('Databricks environment detected. Installing PyNNLF from local repo and full model dependencies...')
    pip_install(['--ignore-requires-python', '--no-deps', '-e', str(REPO_ROOT)])
    deps = [
        'typing_extensions==4.12.2',
        'PyYAML==6.0.2',
        'dill==0.3.9',
        'numpy==2.1.1',
        'pandas==2.2.3',
        'matplotlib==3.9.2',
        'scikit-learn==1.6.0',
        'scipy==1.14.1',
        'statsmodels==0.14.4',
        'joblib==1.4.2',
        'xgboost==3.0.0',
        'torch==2.5.1',
        'prophet==1.1.6',
        'cmdstanpy==1.2.5',
        'holidays==0.72',
    ]
    pip_install(['--upgrade', *deps])
    importlib.invalidate_caches()
    print('Install step complete. If Databricks prompts for a Python restart, restart and rerun from the top.')
else:
    print('Skipping dependency installation.')


## 3. Reserve CPU Cores For Other Work

PyTorch and the tree ensembles use every core by default, which makes the machine sluggish for the hours the heavy models run. Leaving a few cores free costs some run time (the thread-bound models slow roughly in proportion) but keeps video calls and the browser usable. Set `RESERVED_CORES = 0` for a dedicated machine such as a Databricks cluster. This cell must run before the model libraries are imported.

In [ ]:
RESERVED_CORES = 4  # cores left free for other work; 0 for a dedicated run

available = os.cpu_count() or 1
threads = max(1, available - RESERVED_CORES)
for variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[variable] = str(threads)

import torch

torch.set_num_threads(threads)
print(f"{available} cores on this machine; models will use {threads} threads")


## 4. Load The Batch Spec And Show The Plan

The runner script is imported from `scripts/` so the notebook and the terminal share one implementation. The dry run lists every combination and whether it is already done.

In [ ]:
import importlib.util

import pandas as pd
import yaml

runner_path = PROJECT_DIR / 'scripts' / 'run_model_comparison_batch.py'
spec = importlib.util.spec_from_file_location('run_model_comparison_batch', runner_path)
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)

batch = yaml.safe_load(runner.BATCH_PATH.read_text(encoding='utf-8'))
print(f'Batch spec: {runner.BATCH_PATH.relative_to(PROJECT_DIR)}')
print(f'Output folder: {runner.OUTPUT_ROOT.relative_to(PROJECT_DIR)}')
print(f'Datasets: {batch["datasets"]} | horizon: {batch["forecast_horizons"]} | models: {len(batch["model_and_hp"])}')
runner.main(dry_run=True)


## 5. Validate Input Datasets

Every dataset in the spec must resolve to exactly one file in the publication data folder, with no duplicate timestamps and no missing target values.

In [ ]:
from pynnlf.discovery import discover_dataset_path

summaries = []
for dataset_id in batch['datasets']:
    path = discover_dataset_path(PROJECT_DIR / 'data', dataset_id)
    frame = pd.read_csv(path, parse_dates=['datetime'])
    if frame['datetime'].duplicated().any():
        raise ValueError(f'{dataset_id} has duplicate timestamps')
    if frame['netload_kW'].isna().any():
        raise ValueError(f'{dataset_id} has missing netload_kW')
    summaries.append({
        'dataset_id': dataset_id,
        'filename': Path(path).name,
        'rows': len(frame),
        'start': frame['datetime'].min(),
        'end': frame['datetime'].max(),
        'mean_netload_kW': round(frame['netload_kW'].mean(), 1),
        'min_netload_kW': round(frame['netload_kW'].min(), 1),
        'max_netload_kW': round(frame['netload_kW'].max(), 1),
    })

display(pd.DataFrame(summaries))


## 6. Run Every Model Except The LSTM

The 39 pending runs for the 13 other models, about 8 to 10 hours on a 12-thread laptop. If the run is interrupted, rerun this cell: a combination counts as done only once its `a1` summary exists, so at most the interrupted combination is repeated.

To keep the result folder small, the runner switches model pickling off and deletes each experiment's per-fold training series once the experiment has finished; the `a1`, `a2`, `a3` summaries and every test fold stay on disk, and only the summaries and the first test fold are tracked by Git.

In [ ]:
runner.main(skip={'m13'})


## 7. Run The LSTM

The four LSTM runs, about 10 hours on a 12-thread laptop (the ACAP-EDP run alone is nearly 7 of them). Run this cell whenever convenient; it is independent of the cell above and equally resumable.

In [ ]:
runner.main(only={'m13'})


## 8. Validate The Combined Recap

Confirms that the two result folders together hold every combination in the spec exactly once at the 1-day horizon, which is what the section-01 processing notebook needs.

In [ ]:
expected = {(str(ds), str(m), str(hp)) for ds in batch['datasets'] for m, hp in batch['model_and_hp']}
done = runner.completed_keys(runner.RESULT_STORES)
missing = sorted(expected - done)
if missing:
    raise ValueError(f'{len(missing)} combinations are still missing at the 1-day horizon: {missing}')

recap = pd.read_csv(runner.OUTPUT_ROOT / 'a1_experiment_result.csv')
recap = recap.loc[recap['dataset_no'].astype(str).isin([str(ds) for ds in batch['datasets']]) & pd.to_numeric(recap['forecast_horizon_min'], errors='coerce').eq(runner.HORIZON_MINUTES)]
print(f'All {len(expected)} combinations present across both result folders; {len(recap)} of them are in the comparison store.')
display(recap[['experiment_no', 'dataset_no', 'model_name', 'runtime_ms', 'test_nRMSE', 'test_nRMSE_stddev']].sort_values(['dataset_no', 'model_name']))
